# Stan Logistic Regression Random Restarts

Runs Stan mean-field ADVI random restarts for the simulated PyMC-tutorial logistic-regression example.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/logistic_regression")
STAN_FILE_NAME = "stan_logistic_regression.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/logistic_regression
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/logistic_regression/stan_logistic_regression.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars.logistic_regression import (
    load_logistic_regression_data,
    plot_logistic_regression_mean_band_single,
    plot_logistic_regression_selected_restarts,
)

data = load_logistic_regression_data()
labels = data["labels"]
dim = len(labels)

stan_data = {
    "N": int(data["x_train"].shape[0]),
    "K": int(data["x_train"].shape[1]),
    "X": data["x_train"].astype(float),
    "y": data["y_train"].astype(int),
}
PARAM_COLUMNS = [f"b[{i}]" for i in range(1, dim + 1)]
LOG_COLUMNS = []
print(f"N training observations: {stan_data['N']}")
print(labels)


N training observations: 175
['Intercept', 'x1', 'x2', 'x1:x2']


In [3]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 20, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "results"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


{'max_iters': 300000,
 'n_restarts': 20,
 'track_every': 10,
 'parallel': True,
 'max_workers': None,
 'keep_outputs': False,
 'results_dir': 'results'}

In [4]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_by_scenario, FOUR_WAY_STAN_SCENARIOS
print('skipped full cmdstan sweep; reusing saved stan_result.pkl')

skipped full cmdstan sweep; reusing saved stan_result.pkl


In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "wb") as f:
    pickle.dump(stan_result, f)
print(f"Saved Stan results to {STAN_RESULT_PKL}")

Saved Stan results to results/stan_result.pkl


In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "rb") as f:
    stan_result = pickle.load(f)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")
print(f"Loaded Stan results from {STAN_RESULT_PKL}")

Loaded Stan results from results/stan_result.pkl


In [6]:
by_scenario = stan_result_by_scenario(stan_result)

default_means, default_stds, default_iterations = by_scenario["default"]
grad100_means, grad100_stds, grad100_iterations = by_scenario["grad100_default_rest"]
adapt_off_means, adapt_off_stds, adapt_off_iterations = by_scenario["grad100_adapt_off"]
tol_means, tol_stds, tol_iterations = by_scenario["grad100_tol_rel_obj1e-6"]
eta_means, eta_stds, eta_iterations = by_scenario["grad100_tol_rel_obj1e-6_eta0.01"]

import numpy as np

# "matplotlib" (default, static, savable via plt.savefig) or "plotly"
# (interactive: zoom/pan/hover, shown inline). Flip this and rerun the
# plotting cell below to switch how every plot in this notebook renders.
PLOT_BACKEND = "plotly"


In [7]:
reference_file = Path("best_variational_reference.csv")
if not reference_file.exists():
    raise FileNotFoundError("Run compute_best_reference.py after generating processed restart files.")

best_reference = pd.read_csv(reference_file)
reference_means = {
    row["parameter"]: [(float(row["mean"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}
reference_stds = {
    row["parameter"]: [(float(row["sd"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}


def downsample_and_extend(arr, factor, target_len):
    """Downsample along the last (iteration) axis by `factor`, then hold
    the last value flat out to `target_len` points -- a valid projection
    once a run has converged and stopped changing, used here to compare
    the (short) default run against the (long) tolerance-ablation run on
    the latter's own iteration grid.
    """
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled


# 1. Mean +/- 1 SD band across restarts, one combined figure per setup
# (all 4 coefficients as subplot rows), including the plain default.
setups = (
    ("default", default_means, default_stds, default_iterations),
    ("grad100_default_rest", grad100_means, grad100_stds, grad100_iterations),
    ("grad100_adapt_off", adapt_off_means, adapt_off_stds, adapt_off_iterations),
    ("grad100_tol_rel_obj1e-6", tol_means, tol_stds, tol_iterations),
    ("grad100_tol_rel_obj1e-6_eta0.01", eta_means, eta_stds, eta_iterations),
)
for name, means, stds, scen_iterations in setups:
    plot_logistic_regression_mean_band_single(
        means, stds, labels=labels, title_prefix=f"Stan {name} ",
        reference_means=reference_means, reference_stds=reference_stds,
        x=scen_iterations, backend=PLOT_BACKEND,
    )

# 2. Selected-restarts plots: 2 individual restarts of the default vs 2
# of each setup, one combined figure per setup (all 4 coefficients as
# subplot rows). The tolerance-ablation setups run 2x longer (and are
# tracked 10x coarser) than the default run, so we downsample+extend the
# default run's trajectory to align it onto their iteration grid -- see
# downsample_and_extend above.
projected_default_means = downsample_and_extend(default_means, 10, len(tol_iterations))
projected_default_stds = downsample_and_extend(default_stds, 10, len(tol_iterations))
projected_default_means_eta = downsample_and_extend(default_means, 10, len(eta_iterations))
projected_default_stds_eta = downsample_and_extend(default_stds, 10, len(eta_iterations))

comparisons = (
    ("grad100_default_rest", grad100_means, grad100_stds, grad100_iterations, default_means, default_stds),
    ("grad100_adapt_off", adapt_off_means, adapt_off_stds, adapt_off_iterations, default_means, default_stds),
    ("grad100_tol_rel_obj1e-6", tol_means, tol_stds, tol_iterations, projected_default_means, projected_default_stds),
    ("grad100_tol_rel_obj1e-6_eta0.01", eta_means, eta_stds, eta_iterations, projected_default_means_eta, projected_default_stds_eta),
)
for name, means, stds, scen_iterations, d_means, d_stds in comparisons:
    plot_logistic_regression_selected_restarts(
        d_means, d_stds, means, stds, labels=labels, title_prefix=f"Stan {name} ",
        reference_means=reference_means, reference_stds=reference_stds,
        k=2, x=scen_iterations, backend=PLOT_BACKEND,
        single_label="Default (some runs)", multi_label=f"{name} (some runs)",
    )


In [8]:
import pickle
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.logistic_regression import load_logistic_regression_data, plot_logistic_regression_mean_band_single

NB_DIR = REPO_ROOT / "single_MC/logistic_regression"

data = load_logistic_regression_data()
labels = data["labels"]

best_reference = pd.read_csv(NB_DIR / "best_variational_reference.csv")
reference_means = {
    row["parameter"]: [(float(row["mean"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}
reference_stds = {
    row["parameter"]: [(float(row["sd"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}
best_mean_arr = np.array([best_reference.set_index("parameter").loc[lab, "mean"] for lab in labels], dtype=float)
best_std_arr = np.array([best_reference.set_index("parameter").loc[lab, "sd"] for lab in labels], dtype=float)

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

n_td_failures = len(_td_result["failures"])
print(f"tol_1e-6_default_rest: {td_means.shape[0]} restarts, {len(td_iterations)} tracked points, "
      f"final iter={int(td_iterations[-1])}")
print(f"failures: {n_td_failures}")

tol_1e-6_default_rest: 10 restarts, 501 tracked points, final iter=75000
failures: 0


In [9]:
plot_logistic_regression_mean_band_single(
    td_means, td_stds, labels=labels, title_prefix="Stan tol_1e-6_default_rest ",
    reference_means=reference_means, reference_stds=reference_stds,
    x=td_iterations, backend="plotly",
)

In [10]:
import pickle
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.logistic_regression import load_logistic_regression_data, plot_logistic_regression_selected_restarts

NB_DIR = REPO_ROOT / "single_MC/logistic_regression"

data = load_logistic_regression_data()
labels = data["labels"]

best_reference = pd.read_csv(NB_DIR / "best_variational_reference.csv")
reference_means = {
    row["parameter"]: [(float(row["mean"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}
reference_stds = {
    row["parameter"]: [(float(row["sd"]), "best variational approximation")]
    for _, row in best_reference.iterrows()
}

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

def downsample_and_extend(arr, factor, target_len):
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled

projected_default_means = downsample_and_extend(default_means, 15, len(td_iterations))
projected_default_stds = downsample_and_extend(default_stds, 15, len(td_iterations))
plot_logistic_regression_selected_restarts(
    projected_default_means, projected_default_stds, td_means, td_stds, labels=labels,
    title_prefix="Stan tol_1e-6_default_rest ",
    reference_means=reference_means, reference_stds=reference_stds,
    k=2, x=td_iterations, backend="plotly",
    single_label="Default (some runs)", multi_label="tol_1e-6_default_rest (some runs)",
)

In [11]:
import pickle
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.logistic_regression import load_logistic_regression_data

NB_DIR = REPO_ROOT / "single_MC/logistic_regression"

data = load_logistic_regression_data()
labels = data["labels"]

best_reference = pd.read_csv(NB_DIR / "best_variational_reference.csv").set_index("parameter")
best_mean_lookup = np.array([best_reference.loc[lab, "mean"] for lab in labels], dtype=float)
best_std_lookup = np.array([best_reference.loc[lab, "sd"] for lab in labels], dtype=float)

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

import plotly.graph_objects as go
from plotly.subplots import make_subplots

row_labels = labels
n_rows = len(labels)
best_mean_arr = best_mean_lookup
best_std_arr = best_std_lookup

subplot_titles = []
for lab in row_labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx in range(n_rows):
    r = row_idx + 1
    tol_m = tol_means
    td_m = td_means
    show_legend = row_idx == 0
    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_means[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_means[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=show_legend), row=r, col=1)
    fig.add_hline(y=best_mean_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_stds[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_stds[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=False), row=r, col=2)
    fig.add_hline(y=best_std_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=1)
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="Stan tol_1e-6_default_rest vs grad100_tol_rel_obj1e-6")
fig.show()

In [12]:
CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]

with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)

def pct_error(setup, best):
    return (setup - best) / best * 100.0

def zscore(setup_mean, best_mean, best_std):
    return (setup_mean - best_mean) / best_std

print("===== logistic_regression: percent-error / z-score vs best variational approximation =====")
print(f"reference (best) mean: {np.round(best_mean_arr, 4)}")
print(f"reference (best) std : {np.round(best_std_arr, 4)}\n")

rows = []
for cp in CHECKPOINTS:
    r = _cp_results[cp]
    m = np.asarray(r["means_by_scenario"]["default_single"][0, 0, :], dtype=float)
    s = np.asarray(r["stds_by_scenario"]["default_single"][0, 0, :], dtype=float)
    rows.append((f"default, single restart to {cp}", m, s))

for label, scen_name in [
    ("grad100_default_rest (final)", "grad100_default_rest"),
    ("grad100_adapt_off (final)", "grad100_adapt_off"),
    ("grad100_tol_rel_obj1e-6 (final)", "grad100_tol_rel_obj1e-6"),
    ("grad100_tol_rel_obj1e-6_eta0.01 (final)", "grad100_tol_rel_obj1e-6_eta0.01"),
    ("tol_1e-6_default_rest (final)", "tol_1e-6_default_rest"),
]:
    if scen_name == "tol_1e-6_default_rest":
        means, stds, iters = td_means, td_stds, td_iterations
    else:
        means, stds, iters = _by_scenario[scen_name]
    m = np.nanmean(means[:, -1, :], axis=0)
    s = np.nanmean(stds[:, -1, :], axis=0)
    rows.append((f"{label} (iter={int(iters[-1])})", m, s))

for row_name, m, s in rows:
    pe_mean = pct_error(m, best_mean_arr)
    pe_std = pct_error(s, best_std_arr)
    z = zscore(m, best_mean_arr, best_std_arr)
    for j, lab in enumerate(labels):
        print(f"  {row_name:45s} {lab:10s}  pct_err_mean={pe_mean[j]:9.3f}%  pct_err_std={pe_std[j]:9.3f}%  z={z[j]:7.3f}")

===== logistic_regression: percent-error / z-score vs best variational approximation =====
reference (best) mean: [-0.2145  0.6894 -0.6626  1.6834]
reference (best) std : [0.2191 0.1401 0.173  0.1929]

  default, single restart to 50000              Intercept   pct_err_mean=  -17.535%  pct_err_std=   -3.532%  z=  0.172
  default, single restart to 50000              x1          pct_err_mean=   -0.232%  pct_err_std=   -4.905%  z= -0.011
  default, single restart to 50000              x2          pct_err_mean=  -12.295%  pct_err_std=    1.577%  z=  0.471
  default, single restart to 50000              x1:x2       pct_err_mean=    7.716%  pct_err_std=    4.310%  z=  0.674
  default, single restart to 100000             Intercept   pct_err_mean=  -17.535%  pct_err_std=   -3.532%  z=  0.172
  default, single restart to 100000             x1          pct_err_mean=   -0.232%  pct_err_std=   -4.905%  z= -0.011
  default, single restart to 100000             x2          pct_err_mean=  -12.295% 

In [13]:
import pickle
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.logistic_regression import load_logistic_regression_data

NB_DIR = REPO_ROOT / "single_MC/logistic_regression"

data = load_logistic_regression_data()
labels = data["labels"]

import pandas as pd
best_reference = pd.read_csv(NB_DIR / "best_variational_reference.csv").set_index("parameter")
best_mean_arr = np.array([best_reference.loc[lab, "mean"] for lab in labels], dtype=float)
best_std_arr = np.array([best_reference.loc[lab, "sd"] for lab in labels], dtype=float)

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]
default_trace_mean = default_means[0]
default_trace_std = default_stds[0]

CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]
with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)
cp_means = np.array([np.asarray(_cp_results[cp]["means_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_stds = np.array([np.asarray(_cp_results[cp]["stds_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_iters = np.array(CHECKPOINTS)

n_rows = len(labels)
subplot_titles = []
for lab in labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx, lab in enumerate(labels):
    r = row_idx + 1
    show_legend = row_idx == 0
    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_mean[:, row_idx], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_means[:, row_idx], mode="markers", marker=dict(color="orange", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=show_legend), row=r, col=1)
    fig.add_hline(y=best_mean_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_std[:, row_idx], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_stds[:, row_idx], mode="markers", marker=dict(color="orange", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=False), row=r, col=2)
    fig.add_hline(y=best_std_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=1)
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="logistic_regression — posterior mean/std, default settings out to long iteration horizons")
fig.show()